In [7]:

import importlib.util

for package in ["xgboost", "lightgbm"]:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Installed' if installed else 'Not installed'}")


xgboost: Installed
lightgbm: Installed


In [8]:

import numpy as np
import pandas as pd

from pathlib import Path
import sys
from scipy.stats import loguniform, randint, uniform

from sklearn.svm import SVR
from sklearn.ensemble import (
    GradientBoostingRegressor,
    VotingRegressor
)

from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import RandomizedSearchCV, train_test_split, KFold
from sklearn.metrics import (
    mean_squared_log_error,
    mean_absolute_error,
    mean_squared_error,
    make_scorer
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

sys.path.append(str(Path().resolve().parent))
from src.preprocessing import build_pipeline
from src.feature_engineering import add_engineered_features


In [9]:

DATA_DIR = Path("../data")

if not (DATA_DIR / "train.csv").exists():
    DATA_DIR = Path("data")

train_df = pd.read_csv(DATA_DIR / "train.csv")

X = train_df.drop(columns=["Id", "SalePrice"])
y = train_df["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)


X_train: (1168, 79)
X_valid: (292, 79)


In [10]:

X_train_fe = add_engineered_features(
    X_train,
    include_interactions=True
)

X_valid_fe = add_engineered_features(
    X_valid,
    include_interactions=True
)

print("Training:", X_train_fe.shape)
print("Validation:", X_valid_fe.shape)

assert list(X_train_fe.columns) == list(X_valid_fe.columns)


Training: (1168, 86)
Validation: (292, 86)


In [11]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [12]:

def calculate_rmsle(y_true, y_pred):

    y_pred = np.clip(y_pred, 0, None)

    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )


rmsle_scorer = make_scorer(
    calculate_rmsle,
    greater_is_better=False
)


def make_target_pipeline(estimator, X_reference, scale_numeric):

    pipeline = build_pipeline(
        X_reference,
        estimator,
        scale_numeric=scale_numeric
    )

    return TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )


In [13]:

models_config = {

    "SVR": {
        "estimator": SVR(kernel="rbf"),
        "scale_numeric": True,
        "params": {
            "regressor__model__C": loguniform(1, 50),
            "regressor__model__epsilon": loguniform(0.001, 0.05),
            "regressor__model__gamma": loguniform(0.0001, 0.005)
        }
    },

    "Gradient Boosting": {
        "estimator": GradientBoostingRegressor(
            random_state=42
        ),
        "scale_numeric": False,
        "params": {
            "regressor__model__n_estimators": randint(200, 700),
            "regressor__model__learning_rate": loguniform(0.01, 0.1),
            "regressor__model__max_depth": randint(2, 5),
            "regressor__model__min_samples_leaf": randint(1, 8),
            "regressor__model__subsample": uniform(0.6, 0.4)
        }
    },

    "XGBoost": {
        "estimator": XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            random_state=42,
            n_jobs=1
        ),
        "scale_numeric": False,
        "params": {
            "regressor__model__n_estimators": randint(300, 1000),
            "regressor__model__learning_rate": loguniform(0.01, 0.1),
            "regressor__model__max_depth": randint(2, 6),
            "regressor__model__min_child_weight": loguniform(1, 10),
            "regressor__model__subsample": uniform(0.65, 0.35),
            "regressor__model__colsample_bytree": uniform(0.6, 0.4),
            "regressor__model__reg_lambda": loguniform(0.1, 10)
        }
    },

    "LightGBM": {
        "estimator": LGBMRegressor(
            objective="regression",
            verbosity=-1,
            random_state=42,
            n_jobs=1,
            subsample_freq=1
        ),
        "scale_numeric": False,
        "params": {
            "regressor__model__n_estimators": randint(300, 1000),
            "regressor__model__learning_rate": loguniform(0.01, 0.1),
            "regressor__model__num_leaves": randint(7, 32),
            "regressor__model__max_depth": [-1, 3, 5, 7],
            "regressor__model__min_child_samples": randint(5, 30),
            "regressor__model__subsample": uniform(0.65, 0.35),
            "regressor__model__colsample_bytree": uniform(0.6, 0.4),
            "regressor__model__reg_lambda": loguniform(0.01, 10)
        }
    }
}


In [14]:

def optimize_model(name, config, X_tr, y_tr, X_val, y_val):

    print(f"\nOptimizing: {name}")

    estimator = make_target_pipeline(
        config["estimator"],
        X_tr,
        scale_numeric=config["scale_numeric"]
    )

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=config["params"],
        n_iter=15,
        scoring=rmsle_scorer,
        cv=cv,
        n_jobs=-1,
        random_state=42,
        refit=True,
        verbose=1
    )

    search.fit(X_tr, y_tr)

    best_model = search.best_estimator_

    # Predictions on training and validation sets
    train_predictions = best_model.predict(X_tr)
    valid_predictions = best_model.predict(X_val)

    # Retrieve the standard deviation of the best CV configuration
    best_index = search.best_index_

    results = {
        "Model": name,
        "CV RMSLE": -search.best_score_,
        "CV Std": search.cv_results_["std_test_score"][best_index],
        "Train RMSLE": calculate_rmsle(y_tr, train_predictions),
        "Validation RMSLE": calculate_rmsle(y_val, valid_predictions),
        "Validation MAE": mean_absolute_error(y_val, valid_predictions),
        "Validation RMSE": np.sqrt(
            mean_squared_error(y_val, valid_predictions)
        )
    }

    return search, results


In [15]:

searches = {}
model_results = []

for name, config in models_config.items():

    search, results = optimize_model(
        name,
        config,
        X_train_fe,
        y_train,
        X_valid_fe,
        y_valid
    )

    searches[name] = search
    model_results.append(results)


model_results_df = pd.DataFrame(model_results)

display(
    model_results_df
    .sort_values("CV RMSLE")
    .reset_index(drop=True)
)



Optimizing: SVR
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Optimizing: Gradient Boosting
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Optimizing: XGBoost
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Optimizing: LightGBM
Fitting 5 folds for each of 15 candidates, totalling 75 fits


,Model,CV RMSLE,CV Std,Train RMSLE,Validation RMSLE,Validation MAE,Validation RMSE
0,SVR,0.116709,0.018401,0.085928,0.130445,14385.204173,24574.500119
1,Gradient Boosting,0.121146,0.015272,0.054593,0.135925,15995.728003,27890.551197
2,XGBoost,0.121361,0.013978,0.059089,0.128375,15538.114258,26890.217106
3,LightGBM,0.121434,0.013768,0.039982,0.136175,15739.419394,28823.464611


In [16]:

for name, search in searches.items():

    print(f"\n{name}")
    print("-" * 50)

    print("Best CV RMSLE:", round(-search.best_score_, 6))
    print("Best parameters:")

    for parameter, value in search.best_params_.items():
        print(f"  {parameter}: {value}")



SVR
--------------------------------------------------
Best CV RMSLE: 0.116709
Best parameters:
  regressor__model__C: 5.4182823195332395
  regressor__model__epsilon: 0.003124565071260871
  regressor__model__gamma: 0.001095266274863256

Gradient Boosting
--------------------------------------------------
Best CV RMSLE: 0.121146
Best parameters:
  regressor__model__learning_rate: 0.04597505784732168
  regressor__model__max_depth: 3
  regressor__model__min_samples_leaf: 6
  regressor__model__n_estimators: 589
  regressor__model__subsample: 0.6831766651472755

XGBoost
--------------------------------------------------
Best CV RMSLE: 0.121361
Best parameters:
  regressor__model__colsample_bytree: 0.8281775897621597
  regressor__model__learning_rate: 0.03317678207058294
  regressor__model__max_depth: 3
  regressor__model__min_child_weight: 2.1150972021685583
  regressor__model__n_estimators: 637
  regressor__model__reg_lambda: 1.200561092619211
  regressor__model__subsample: 0.855362907982

In [17]:

ranked_models = (
    model_results_df
    .sort_values("CV RMSLE")["Model"]
    .tolist()
)

top_two_models = ranked_models[:2]

print("Selected models for ensembling:")

for name in top_two_models:
    print(
        f"{name}: "
        f"CV RMSLE = "
        f"{model_results_df.loc[model_results_df['Model'] == name, 'CV RMSLE'].iloc[0]:.6f}"
    )


Selected models for ensembling:
SVR: CV RMSLE = 0.116709
Gradient Boosting: CV RMSLE = 0.121146


In [18]:

individual_predictions = {}

for name in top_two_models:

    individual_predictions[name] = (
        searches[name].best_estimator_.predict(X_valid_fe)
    )

residuals = pd.DataFrame({
    name: y_valid.to_numpy() - predictions
    for name, predictions in individual_predictions.items()
})

display(residuals.corr())


,SVR,Gradient Boosting
SVR,1.000000,0.862303
Gradient Boosting,0.862303,1.000000


In [20]:
from sklearn.model_selection import cross_val_score
base_estimators = [
    ("model_1", searches[top_two_models[0]].best_estimator_),
    ("model_2", searches[top_two_models[1]].best_estimator_)
]

weight_schemes = [
    ("Equal weights", [1, 1]),
    (f"More weight: {top_two_models[0]}", [2, 1]),
    (f"More weight: {top_two_models[1]}", [1, 2])
]

ensemble_results = []
ensemble_models = {}

for name, weights in weight_schemes:

    print(f"Evaluating ensemble: {name}")

    ensemble = VotingRegressor(
        estimators=base_estimators,
        weights=weights,
        n_jobs=1
    )

    # Cross-validation
    scores = cross_val_score(
        ensemble,
        X_train_fe,
        y_train,
        scoring=rmsle_scorer,
        cv=cv,
        n_jobs=-1
    )

    # Fit on the complete training split
    ensemble.fit(X_train_fe, y_train)

    # Validation predictions
    predictions = ensemble.predict(X_valid_fe)

    results = {
        "Ensemble": name,
        "CV RMSLE": -scores.mean(),
        "CV Std": scores.std(),
        "Validation RMSLE": calculate_rmsle(
            y_valid,
            predictions
        ),
        "Validation MAE": mean_absolute_error(
            y_valid,
            predictions
        ),
        "Validation RMSE": np.sqrt(
            mean_squared_error(y_valid, predictions)
        )
    }

    ensemble_results.append(results)
    ensemble_models[name] = ensemble


ensemble_results_df = pd.DataFrame(ensemble_results)

display(
    ensemble_results_df
    .sort_values("CV RMSLE")
    .reset_index(drop=True)
)


Evaluating ensemble: Equal weights
Evaluating ensemble: More weight: SVR
Evaluating ensemble: More weight: Gradient Boosting


,Ensemble,CV RMSLE,CV Std,Validation RMSLE,Validation MAE,Validation RMSE
0,More weight: SVR,0.113513,0.017589,0.129481,14398.009053,24850.484383
1,Equal weights,0.113698,0.017052,0.130070,14609.022710,25317.952870
2,More weight: Gradient Boosting,0.115059,0.016462,0.131356,14937.501018,25991.129975


In [21]:

individual_comparison = model_results_df[
    [
        "Model",
        "CV RMSLE",
        "CV Std",
        "Validation RMSLE",
        "Validation MAE",
        "Validation RMSE"
    ]
].copy()

individual_comparison["Type"] = "Individual"
individual_comparison = individual_comparison.rename(
    columns={"Model": "Candidate"}
)


ensemble_comparison = ensemble_results_df.rename(
    columns={"Ensemble": "Candidate"}
).copy()

ensemble_comparison["Type"] = "Ensemble"


final_comparison = pd.concat(
    [individual_comparison, ensemble_comparison],
    ignore_index=True
)

display(
    final_comparison
    .sort_values("CV RMSLE")
    .reset_index(drop=True)
)


,Candidate,CV RMSLE,CV Std,Validation RMSLE,Validation MAE,Validation RMSE,Type
0,More weight: SVR,0.113513,0.017589,0.129481,14398.009053,24850.484383,Ensemble
1,Equal weights,0.113698,0.017052,0.130070,14609.022710,25317.952870,Ensemble
2,More weight: Gradient Boosting,0.115059,0.016462,0.131356,14937.501018,25991.129975,Ensemble
3,SVR,0.116709,0.018401,0.130445,14385.204173,24574.500119,Individual
4,Gradient Boosting,0.121146,0.015272,0.135925,15995.728003,27890.551197,Individual
5,XGBoost,0.121361,0.013978,0.128375,15538.114258,26890.217106,Individual
6,LightGBM,0.121434,0.013768,0.136175,15739.419394,28823.464611,Individual


In [22]:

for name in searches:

    print(f"\n{name}")
    print("=" * 60)

    print(f"Best CV RMSLE: {-searches[name].best_score_:.6f}")

    for parameter, value in searches[name].best_params_.items():
        print(f"{parameter}: {value}")


print("\nSelected ensemble: More weight: SVR")
print("Weights: SVR = 2, Gradient Boosting = 1")



SVR
Best CV RMSLE: 0.116709
regressor__model__C: 5.4182823195332395
regressor__model__epsilon: 0.003124565071260871
regressor__model__gamma: 0.001095266274863256

Gradient Boosting
Best CV RMSLE: 0.121146
regressor__model__learning_rate: 0.04597505784732168
regressor__model__max_depth: 3
regressor__model__min_samples_leaf: 6
regressor__model__n_estimators: 589
regressor__model__subsample: 0.6831766651472755

XGBoost
Best CV RMSLE: 0.121361
regressor__model__colsample_bytree: 0.8281775897621597
regressor__model__learning_rate: 0.03317678207058294
regressor__model__max_depth: 3
regressor__model__min_child_weight: 2.1150972021685583
regressor__model__n_estimators: 637
regressor__model__reg_lambda: 1.200561092619211
regressor__model__subsample: 0.8553629079823468

LightGBM
Best CV RMSLE: 0.121434
regressor__model__colsample_bytree: 0.6092249700165663
regressor__model__learning_rate: 0.03347916827555514
regressor__model__max_depth: 5
regressor__model__min_child_samples: 14
regressor__model


# Phase 7 — Model Optimization and Ensembles: Conclusions

Four algorithms were evaluated using the engineered dataset:
SVR, Gradient Boosting, XGBoost and LightGBM.

SVR achieved the best individual cross-validation performance,
while XGBoost achieved the best holdout validation RMSLE.

An ensemble combining SVR and Gradient Boosting was then evaluated
using different prediction weights.

The best configuration assigned twice as much weight to SVR
as to Gradient Boosting.

## Final comparison

| Candidate | CV RMSLE | Validation RMSLE |
|---|---:|---:|
| SVR + Gradient Boosting (2:1) | 0.113513 | 0.129481 |
| Equal weights | 0.113698 | 0.130070 |
| SVR individual | 0.116709 | 0.130445 |
| XGBoost individual | 0.121361 | 0.128375 |

## Selected candidate

The SVR + Gradient Boosting ensemble with weights [2, 1]
was selected as the leading candidate based on its cross-validation
performance and its improvement over the individual SVR model.

XGBoost was retained as an alternative because of its strong
holdout validation result.

The selected model will be evaluated on the Kaggle test dataset
in the final training and submission phase.

Further evaluation is necessary because the validation set has
been reused during model experimentation.
